# Lab 2 - Representação

1. TF-IDF
2. Embeddings (Word2Vec)
3. Embeddings (Glove)
4. Embeddings Fast Text
5. BERT
6. Sentence BERT
7. Hugging Face (BERT e suas Variantes)







## Drive e Bibliotecas

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [3]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 56.8 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from nltk.corpus import stopwords
import nltk
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from tqdm import tqdm
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.manifold import TSNE



## Leitura dos comentários (Google Drive)

### Os arquivos possuem 25k linhas cada. Para processar de forma mais rápida, mude o RunTime Type para T4 GPU. Usando menos exemplos (n_samples) o processamento é mais rápido.

In [5]:
n_samples = 5000

# Carregamento dos dados
train_df_full = pd.read_csv("/content/drive/MyDrive/Data/IMDB_Small/comments_train.txt")
test_df_full  = pd.read_csv("/content/drive/MyDrive/Data/IMDB_Small/comments_test.txt")

# Criação do subset usando .sample()
# random_state garante que a amostra seja a mesma toda vez que você rodar o código
train_df = train_df_full.sample(n=n_samples, random_state=42)
test_df = test_df_full.sample(n=n_samples, random_state=42)

# Verificação
print(f"Tamanho do treino: {len(train_df)}")
print(f"Tamanho do teste: {len(test_df)}")



Tamanho do treino: 5000
Tamanho do teste: 5000


## 1 - TF-IDF

### Carregas as StopWords e os comentários

In [6]:
# --------------------------------------------------
# Download stopwords
# --------------------------------------------------
nltk.download('stopwords')
stop_words_en = stopwords.words('english')

# --------------------------------------------------
# Dados
# --------------------------------------------------
X_train_texts = train_df["review"].astype(str)
y_train = train_df["label"]

X_test_texts = test_df["review"].astype(str)
y_test = test_df["label"]


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


### Treina o TF-IDF

In [7]:
# --------------------------------------------------
# Parametros do TF-IDF
# --------------------------------------------------
vectorizer = TfidfVectorizer(
	stop_words=stop_words_en,
	max_features=1000,
	ngram_range=(1,2), #(min,max) (1,2) -> unigramas e bigramas
	lowercase=True,
)

X_train = vectorizer.fit_transform(X_train_texts)
X_test  = vectorizer.transform(X_test_texts)

#feature_names = vectorizer.get_feature_names_out()


### Classificação a representação TF-IDF com uma Regrassão Logistica

In [8]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("\nResultados:")
print(classification_report(y_test, predictions))
print(confusion_matrix(y_test, predictions))



Resultados:
              precision    recall  f1-score   support

         neg       0.85      0.84      0.85      2515
         pos       0.84      0.85      0.85      2485

    accuracy                           0.85      5000
   macro avg       0.85      0.85      0.85      5000
weighted avg       0.85      0.85      0.85      5000

[[2124  391]
 [ 374 2111]]


### t-SNE para Visualização das representações

## 2 - Embedding (word2vec)

### Treinamento da rede neural

In [9]:
from gensim.models.callbacks import CallbackAny2Vec

VECTOR_SIZE = 100
WINDOW = 5
MIN_COUNT = 5  # remove palavras raras
WORKERS = 4
EPOCHS = 20

LABEL_MAP = {"pos": 1, "neg": 0}

class LossLogger(CallbackAny2Vec):
    def __init__(self):
        self.epoch = 0
        self.loss_previous = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        loss_diff = loss - self.loss_previous

        print(f"Epoch {self.epoch} | Loss: {loss_diff:.2f}")

        self.loss_previous = loss
        self.epoch += 1


# -----------------------------
# Função para vetor médio
# -----------------------------
def document_vector(doc, model):
    words = [w for w in doc if w in model.wv]
    if len(words) == 0:
        return np.zeros(model.vector_size)
    return np.mean(model.wv[words], axis=0)

# -----------------------------
# Ler dados
# -----------------------------
#def load_data(csv_file):
#    df = pd.read_csv(csv_file)
#    df = df[["review", "label"]]
#    df["label"] = df["label"].map(LABEL_MAP)
#    df = df.dropna()
#    return df

# -----------------------------
# Tokenização
# -----------------------------
train_tokens = [simple_preprocess(text) for text in train_df["review"]]
test_tokens = [simple_preprocess(text) for text in test_df["review"]]

# -----------------------------
# Treinar a Rede Neural Word2Vec
# -----------------------------
print("Treinando Word2Vec...")
w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=WORKERS,
    epochs=EPOCHS,
    sg = 0, # 0 - CBOW, 1- SkipGram
    compute_loss=True,
    callbacks=[LossLogger()]
)



Treinando Word2Vec...
Epoch 0 | Loss: 125607.73
Epoch 1 | Loss: 110452.23
Epoch 2 | Loss: 106795.28
Epoch 3 | Loss: 118088.47
Epoch 4 | Loss: 112476.72
Epoch 5 | Loss: 81840.50
Epoch 6 | Loss: 130703.00
Epoch 7 | Loss: 77881.62
Epoch 8 | Loss: 95597.88
Epoch 9 | Loss: 93261.81
Epoch 10 | Loss: 76779.50
Epoch 11 | Loss: 77182.88
Epoch 12 | Loss: 91484.62
Epoch 13 | Loss: 60354.50
Epoch 14 | Loss: 86729.12
Epoch 15 | Loss: 74955.75
Epoch 16 | Loss: 71092.38
Epoch 17 | Loss: 89171.75
Epoch 18 | Loss: 117596.50
Epoch 19 | Loss: 72781.38


### Cria as representações

In [10]:
# -----------------------------
# Gerar embeddings
# -----------------------------
print("Gerando embeddings de treino...")
X_train = np.array([
    document_vector(doc, w2v_model)
    for doc in tqdm(train_tokens)
])

print("Gerando embeddings de teste...")
X_test = np.array([
    document_vector(doc, w2v_model)
    for doc in tqdm(test_tokens)
])

y_train = train_df["label"].values
y_test = test_df["label"].values

Gerando embeddings de treino...


100%|██████████| 1000/1000 [00:00<00:00, 1910.98it/s]


Gerando embeddings de teste...


100%|██████████| 1000/1000 [00:00<00:00, 1676.04it/s]


### Classificação a representação word2vec com uma Regressão Logistica

In [11]:

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\nResultados:")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))



Resultados:
              precision    recall  f1-score   support

         neg       0.69      0.72      0.71       511
         pos       0.70      0.66      0.68       489

    accuracy                           0.69      1000
   macro avg       0.69      0.69      0.69      1000
weighted avg       0.69      0.69      0.69      1000

[[370 141]
 [167 322]]


## 3 - GLOVE

In [12]:
import gensim.downloader as api
from gensim.models import KeyedVectors

# 1. Baixar o modelo GloVe (treinado no Twitter ou Wikipedia)
# 'glove-wiki-gigaword-100' tem 100 dimensões e foi treinado na Wikipedia
print("Baixando/Carregando modelo GloVe... Isso pode demorar um pouco.")
glove_model = api.load("glove-wiki-gigaword-100")

# 2. Testando a Similaridade
palavra = "university"
print(f"\nPalavras mais próximas de '{palavra}':")
for sim_word, score in glove_model.most_similar(palavra, topn=5):
    print(f" - {sim_word}: {score:.4f}")

# 3. Resolvendo Analogias (O famoso exemplo: King - Man + Woman = Queen)
# Lógica: "A está para B, assim como C está para ???"
print("\nResolvendo analogia: king - man + woman = ?")
resultado = glove_model.most_similar(positive=['king', 'woman'], negative=['man'], topn=1)
print(f"Resultado: {resultado[0][0]}")

# 4. Encontrando a 'Infiltrada' (Qual palavra não pertence ao grupo?)
grupo = ["breakfast", "cereal", "dinner", "lunch", "bicycle"]
infiltrada = glove_model.doesnt_match(grupo)
print(f"\nNo grupo {grupo}, a palavra que não combina é: {infiltrada}")

Baixando/Carregando modelo GloVe... Isso pode demorar um pouco.
[==================================================] 100.0% 128.1/128.1MB downloaded

Palavras mais próximas de 'university':
 - college: 0.8294
 - harvard: 0.8156
 - yale: 0.8114
 - professor: 0.8104
 - graduate: 0.7993

Resolvendo analogia: king - man + woman = ?
Resultado: queen

No grupo ['breakfast', 'cereal', 'dinner', 'lunch', 'bicycle'], a palavra que não combina é: bicycle


In [13]:
def get_document_vector(tokens, model):
    # Filtra palavras que existem no vocabulário do GloVe
    vectors = [model[word] for word in tokens if word in model]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    # Retorna a média aritmética dos vetores das palavras
    return np.mean(vectors, axis=0)

# Tokenização básica (similar ao que você já estava fazendo)
train_tokens = [simple_preprocess(text) for text in train_df["review"]]
test_tokens  = [simple_preprocess(text) for text in test_df["review"]]

# Criando as matrizes de características (X)
X_train = np.array([get_document_vector(tokens, glove_model) for tokens in train_tokens])
X_test  = np.array([get_document_vector(tokens, glove_model) for tokens in test_tokens])

# Extraindo os rótulos (y)
y_train = train_df["label"]
y_test  = test_df["label"]

In [14]:
# Inicializa e treina o modelo
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Predição e Avaliação
y_pred = clf.predict(X_test)

print("\nResultados:")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))



Resultados:
              precision    recall  f1-score   support

         neg       0.75      0.83      0.79       511
         pos       0.80      0.71      0.75       489

    accuracy                           0.77      1000
   macro avg       0.78      0.77      0.77      1000
weighted avg       0.77      0.77      0.77      1000

[[424  87]
 [141 348]]


## 4 - FastText

In [15]:
from gensim.models import FastText
from gensim.utils import simple_preprocess
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 1. Tokenização
train_tokens = [simple_preprocess(text) for text in train_df["review"]]
test_tokens  = [simple_preprocess(text) for text in test_df["review"]]

# 2. Treinamento do Modelo FastText
print("Treinando FastText...")
ft_model = FastText(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=5,
    sg=0, # 1 para Skip-gram (mais lento, porém melhor para palavras raras)
    epochs=10
)

Treinando FastText...


In [16]:
def get_document_vector_ft(tokens, model):
    # O FastText sempre terá um vetor, mesmo para palavras "Out-of-Vocabulary"
    vectors = [model.wv[word] for word in tokens]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

# Criando as matrizes X
X_train_ft = np.array([get_document_vector_ft(tokens, ft_model) for tokens in train_tokens])
X_test_ft  = np.array([get_document_vector_ft(tokens, ft_model) for tokens in test_tokens])

# Treinando a Regressão Logística
clf_ft = LogisticRegression(max_iter=1000)
clf_ft.fit(X_train_ft, train_df["label"])

# Avaliação
y_pred_ft = clf_ft.predict(X_test_ft)
print("\nRelatório FastText:")
print(classification_report(test_df["label"], y_pred_ft))






Relatório FastText:
              precision    recall  f1-score   support

         neg       0.63      0.70      0.66       511
         pos       0.65      0.57      0.61       489

    accuracy                           0.64      1000
   macro avg       0.64      0.64      0.64      1000
weighted avg       0.64      0.64      0.64      1000



## 5 - BERT

In [6]:
import torch

#MODEL_NAME = "distilbert-base-uncased". # modelo mais simples
MODEL_NAME = "roberta-base" # modelo mais complexo
MAX_LENGTH = 512 # 128, 256, 512. Quanto maior mais tempo de processamento
BATCH_SIZE = 32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

# -----------------------------
# Função para extrair embeddings
# -----------------------------
def get_embeddings(texts):
	embeddings = []

	for i in tqdm(range(0, len(texts), BATCH_SIZE)):
		batch = texts[i:i+BATCH_SIZE]

		inputs = tokenizer(
			batch,
			padding=True,
			truncation=True,
			max_length=MAX_LENGTH,
			return_tensors="pt"
		).to(device)

		with torch.no_grad():
			outputs = model(**inputs)

		# usar embedding do CLS token
		cls_embeddings = outputs.last_hidden_state[:, 0, :]

		embeddings.append(cls_embeddings.cpu().numpy())

	return np.vstack(embeddings)

# -----------------------------
# Gerar features
# -----------------------------
print("Extraindo embeddings BERT...")

X_train = get_embeddings(train_df["review"].tolist())
X_test = get_embeddings(test_df["review"].tolist())

y_train = train_df["label"].values
y_test = test_df["label"].values

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Extraindo embeddings BERT...


100%|██████████| 32/32 [00:25<00:00,  1.24it/s]


In [7]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Predição e Avaliação
y_pred = clf.predict(X_test)

print("\nResultados:")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))



Resultados:
              precision    recall  f1-score   support

         neg       0.85      0.87      0.86       511
         pos       0.86      0.84      0.85       489

    accuracy                           0.85      1000
   macro avg       0.86      0.85      0.85      1000
weighted avg       0.86      0.85      0.85      1000

[[445  66]
 [ 79 410]]


## 6 - Sentece BERT (SBERT)

In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# -----------------------------
# 1. Configurações e Modelo
# -----------------------------
# all-mpnet-base-v2: Mais lento, porém muito mais preciso
# all-MiniLM-L6-v2: Muito rápido,
MODEL_NAME = "all-mpnet-base-v2"
model = SentenceTransformer(MODEL_NAME)

# -----------------------------
# 2. Função para Gerar Embeddings em Lote
# -----------------------------
def encode_dataframe(df):
    print(f"Gerando embeddings para {len(df)} registros...")
    # O SentenceTransformer já lida internamente com lotes (batches)
    # e possui barra de progresso nativa (show_progress_bar)
    embeddings = model.encode(
        df["review"].tolist(),
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    return np.array(embeddings)

# -----------------------------
# 3. Processamento
# -----------------------------
# Gerar características (X)
X_train = encode_dataframe(train_df)
X_test = encode_dataframe(test_df)

# Extrair rótulos (y) - Assumindo que seu DF já tem 0/1 ou pos/neg
# Se precisar mapear, use: .map({"pos": 1, "neg": 0})
y_train = train_df["label"]
y_test = test_df["label"]

# -----------------------------
# 4. Classificação
# -----------------------------
print("\nTreinando Regressão Logística...")
clf = LogisticRegression(max_iter=1000, C=1.0)
clf.fit(X_train, y_train)

# 5. Avaliação
y_pred = clf.predict(X_test)
print("\nResultado Final com SBERT + Logistic Regression:")
print(classification_report(y_test, y_pred))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

Gerando embeddings para 1000 registros...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Gerando embeddings para 1000 registros...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]


Treinando Regressão Logística...

Resultado Final com SBERT + Logistic Regression:
              precision    recall  f1-score   support

         neg       0.84      0.86      0.85       511
         pos       0.85      0.83      0.84       489

    accuracy                           0.84      1000
   macro avg       0.84      0.84      0.84      1000
weighted avg       0.84      0.84      0.84      1000



## 7 - Hugging Face

In [13]:
from transformers import pipeline

# Este modelo é uma versão destilada (leve) do BERT, treinada para sentimentos
specific_model = "distilbert-base-uncased-finetuned-sst-2-english"
classifier = pipeline("sentiment-analysis", model=specific_model, device=0) # device=0 usa GPU

# Testando um comentário
result = classifier("This movie is a absolute masterpiece, though a bit long.")
print(result)
# Saída: [{'label': 'POSITIVE', 'score': 0.9998}]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9995498061180115}]


In [14]:
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from tqdm import tqdm
import torch

# ---------------------------------------------------------
# Configurações
# ---------------------------------------------------------
MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"
BATCH_SIZE = 8
NUM_COMENTARIOS = 100

VERBOSE = True   #  controle de impressão

# ---------------------------------------------------------
# Normalização
# ---------------------------------------------------------
def normalize_label(label):
    if isinstance(label, str):
        label = label.lower().strip()
        if label in ["positive", "pos", "1"]:
            return 1
        elif label in ["negative", "neg", "0"]:
            return 0
    elif isinstance(label, (int, float)):
        return int(label)
    raise ValueError(f"Label desconhecido: {label}")

# ---------------------------------------------------------
# Inicialização
# ---------------------------------------------------------
classifier = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME,
    device=-1
)

# ---------------------------------------------------------
# Dados
# ---------------------------------------------------------
df_sample = test_df.sample(NUM_COMENTARIOS, random_state=42)

comentarios = df_sample["review"].tolist()
labels_reais = [normalize_label(x) for x in df_sample["label"].tolist()]

print(f"Total: {len(comentarios)}")

# ---------------------------------------------------------
# Inferência
# ---------------------------------------------------------
y_pred_numeric = []

print("Classificando...")

for i in tqdm(range(0, len(comentarios), BATCH_SIZE)):
    batch = comentarios[i:i+BATCH_SIZE]
    outputs = classifier(batch, truncation=True)

    for j, out in enumerate(outputs):
        idx = i + j

        label_pred = normalize_label(out["label"])
        label_real = labels_reais[idx]
        score = out["score"]

        y_pred_numeric.append(label_pred)

        # 🔥 impressão opcional
        if VERBOSE:
            status = "✅" if label_pred == label_real else "❌"
            texto_curto = comentarios[idx][:80].replace("\n", " ")

            pred_str = "POSITIVE" if label_pred == 1 else "NEGATIVE"
            real_str = "POSITIVE" if label_real == 1 else "NEGATIVE"

            print(f"[{idx:04d}] {status} | Pred: {pred_str} ({score:.2%}) | Real: {real_str} | {texto_curto}")

# ---------------------------------------------------------
# Métricas
# ---------------------------------------------------------
acuracia = accuracy_score(labels_reais, y_pred_numeric)
cm = confusion_matrix(labels_reais, y_pred_numeric)

print("\n" + "="*60)
print(f"ACURÁCIA: {acuracia:.4f}")
print("="*60)

print("\nMATRIZ DE CONFUSÃO:")
print(cm)

print("\nRELATÓRIO:")
print(classification_report(labels_reais, y_pred_numeric, target_names=["NEGATIVE", "POSITIVE"]))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Total: 100
Classificando...


  8%|▊         | 1/13 [00:01<00:22,  1.87s/it]

[0000] ✅ | Pred: POSITIVE (85.70%) | Real: POSITIVE | Fuckland is an interesting film. I personally love the Dogma movement. I wish it
[0001] ❌ | Pred: NEGATIVE (98.52%) | Real: POSITIVE | Out of all the parodies of Star Wars I've seen, this is probably the funniest. N
[0002] ✅ | Pred: NEGATIVE (99.95%) | Real: NEGATIVE | At first I thought that this one was supposed to be somewhat of a comedy/horror 
[0003] ❌ | Pred: NEGATIVE (99.67%) | Real: POSITIVE | I just saw this movie in a sneak preview and before reading my comment you have 
[0004] ✅ | Pred: NEGATIVE (99.25%) | Real: NEGATIVE | This movie was so dumb and slow was it ever slow. The only good part of the film
[0005] ❌ | Pred: POSITIVE (99.67%) | Real: NEGATIVE | I caught this film -- under the title of "What Lies Above" -- on Lifetime movie 
[0006] ❌ | Pred: NEGATIVE (96.98%) | Real: POSITIVE | This new installment to the Child's Play series has not one scary scene but tons
[0007] ✅ | Pred: NEGATIVE (99.96%) | Real: NEGATIVE | I

 15%|█▌        | 2/13 [00:03<00:21,  1.98s/it]

...

[0080] ✅ | Pred: POSITIVE (99.95%) | Real: POSITIVE | honestly.. this show warms my heart, i watch it EVERYDAY on fox family and now t
[0081] ✅ | Pred: NEGATIVE (99.62%) | Real: NEGATIVE | Yes, this bizarre feature was written by John Sayles. Shot in Toronto, it's yet 
[0082] ❌ | Pred: NEGATIVE (99.82%) | Real: POSITIVE | Nothing like this was seen on TV at that time and probably never will again. Fro
[0083] ✅ | Pred: POSITIVE (99.30%) | Real: POSITIVE | First of all, before I start my review, I just read every review for 'The Muppet
[0084] ✅ | Pred: NEGATIVE (99.98%) | Real: NEGATIVE | I am a fan of his ... This movie sucked really bad. Even worse than Ticker! & Th
[0085] ✅ | Pred: NEGATIVE (99.63%) | Real: NEGATIVE | That's the only word I can think of to describe this movie. Not waste as in a wa
[0086] ✅ | Pred: NEGATIVE (99.97%) | Real: NEGATIVE | Heart of Darkness Movie Review Could a book that is well known for its eloquent 
[0087] ✅ | Pred: NEGATIVE (99.64%) | Real: NEGATIVE | W

 92%|█████████▏| 12/13 [00:24<00:01,  1.98s/it]

[0088] ✅ | Pred: POSITIVE (99.05%) | Real: POSITIVE | "Subspecies," like many other horror films, gets a raw deal on IMDb. The majorit
[0089] ✅ | Pred: NEGATIVE (96.34%) | Real: NEGATIVE | Copy cats have copied this movie from a 1974 Hindi movie called "Call Girl"! "Ca
[0090] ❌ | Pred: NEGATIVE (97.24%) | Real: POSITIVE | The film someone had to make.<br /><br />Waco: The Rules of Engagement dissects 
[0091] ❌ | Pred: NEGATIVE (96.51%) | Real: POSITIVE | Given the acting roles he played in the 1940s (Casper Gutman, Signior Ferrari, M
[0092] ✅ | Pred: POSITIVE (99.97%) | Real: POSITIVE | This is by far one of the best movies I have seen in a very long time. Top 20 of
[0093] ✅ | Pred: NEGATIVE (99.97%) | Real: NEGATIVE | Latest attempt to revive the series actually based on a pretty good idea but wit
[0094] ✅ | Pred: NEGATIVE (99.94%) | Real: NEGATIVE | This documentary is rife with problems.<br /><br />How arrogant is it to make a 
[0095] ❌ | Pred: NEGATIVE (95.39%) | Real: POSITIVE | T

100%|██████████| 13/13 [00:25<00:00,  1.99s/it]

[0096] ❌ | Pred: NEGATIVE (97.79%) | Real: POSITIVE | Sandra Bullock paints a believable picture as the troubled detective, though see
[0097] ✅ | Pred: POSITIVE (88.23%) | Real: POSITIVE | Martin Ritt's first film offers an exceptional existentialist answer (three year
[0098] ✅ | Pred: POSITIVE (99.35%) | Real: POSITIVE | If you haven't seen this film, do it. Its a genremix as i've never seen another.
[0099] ✅ | Pred: POSITIVE (93.04%) | Real: POSITIVE | Does exactly what you expect, and then some. The first movie, was a step up from

ACURÁCIA: 0.8000

MATRIZ DE CONFUSÃO:
[[41  5]
 [15 39]]

RELATÓRIO:
              precision    recall  f1-score   support

    NEGATIVE       0.73      0.89      0.80        46
    POSITIVE       0.89      0.72      0.80        54

    accuracy                           0.80       100
   macro avg       0.81      0.81      0.80       100
weighted avg       0.82      0.80      0.80       100

